# 01 — Generateur de plaques d'immatriculation francaises

Ce notebook genere des images synthetiques de plaques SIV (AA-123-AA) pour creer notre dataset d'entrainement.

**Output :** 10 000 images de plaques nettes (520x110 px)

**Runtime :** CPU suffit (~30 min pour 10K images)

## 1. Setup

In [ ]:
!pip install -q pillow numpy matplotlib
!mkdir -p dataset/clean dataset/degraded fonts

In [ ]:
# Telecharger une police compatible plaques FR
# Option 1 : police libre qui ressemble au format SIV
# La police officielle "Numberplate France" est sur maisfontes.com
# Pour le notebook on utilise une police monospace bold qui fonctionne bien
import urllib.request
import os

# Liberation Mono Bold — police libre, bonne lisibilite pour les plaques
FONT_URL = "https://github.com/liberationfonts/liberation-fonts/files/7261482/liberation-fonts-ttf-2.1.5.tar.gz"
if not os.path.exists('fonts/LiberationMono-Bold.ttf'):
    urllib.request.urlretrieve(FONT_URL, 'fonts/liberation.tar.gz')
    !cd fonts && tar xzf liberation.tar.gz --strip-components=1 '*/LiberationMono-Bold.ttf'
    !ls fonts/*.ttf

# Si tu as telecharge "Numberplate France" (.ttf) depuis maisfontes.com,
# uploade-la dans le dossier fonts/ et change FONT_PATH ci-dessous
FONT_PATH = 'fonts/LiberationMono-Bold.ttf'
print(f'Police : {FONT_PATH}')

## 2. Generateur de texte SIV

In [ ]:
import random
import string

# Regles SIV : AA-123-AA
# - Lettres : A-Z sauf I, O, U (confusion avec 1, 0, V)
# - Chiffres : 001 a 999
LETTRES_SIV = [c for c in string.ascii_uppercase if c not in 'IOU']

def generer_texte_plaque():
    """Genere un texte de plaque SIV aleatoire (ex: AB-123-CD)."""
    l1 = random.choice(LETTRES_SIV) + random.choice(LETTRES_SIV)
    num = f"{random.randint(1, 999):03d}"
    l2 = random.choice(LETTRES_SIV) + random.choice(LETTRES_SIV)
    return f"{l1}-{num}-{l2}"

# Exemples
for _ in range(5):
    print(generer_texte_plaque())

## 3. Renderer de plaque

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Dimensions (pixels, ratio officiel 520mm x 110mm)
PLATE_W, PLATE_H = 520, 110
BAND_W = 45  # largeur bande bleue
CORNER_RADIUS = 8

# Couleurs
WHITE = (255, 255, 255)
BLACK = (20, 20, 20)
BLUE_EU = (0, 51, 153)       # bleu bande europeenne
BLUE_REGION = (0, 82, 155)   # bleu bande region
YELLOW_STARS = (255, 204, 0) # etoiles EU


def draw_rounded_rect(draw, xy, radius, fill):
    """Dessine un rectangle aux coins arrondis."""
    x0, y0, x1, y1 = xy
    draw.rectangle([x0 + radius, y0, x1 - radius, y1], fill=fill)
    draw.rectangle([x0, y0 + radius, x1, y1 - radius], fill=fill)
    draw.pieslice([x0, y0, x0 + 2*radius, y0 + 2*radius], 180, 270, fill=fill)
    draw.pieslice([x1 - 2*radius, y0, x1, y0 + 2*radius], 270, 360, fill=fill)
    draw.pieslice([x0, y1 - 2*radius, x0 + 2*radius, y1], 90, 180, fill=fill)
    draw.pieslice([x1 - 2*radius, y1 - 2*radius, x1, y1], 0, 90, fill=fill)


def generer_plaque(texte=None, font_path=FONT_PATH):
    """Genere une image de plaque francaise SIV."""
    if texte is None:
        texte = generer_texte_plaque()

    img = Image.new('RGB', (PLATE_W, PLATE_H), (200, 200, 200))
    draw = ImageDraw.Draw(img)

    # Fond blanc avec coins arrondis
    draw_rounded_rect(draw, (0, 0, PLATE_W-1, PLATE_H-1), CORNER_RADIUS, WHITE)

    # Bordure noire fine
    draw.rounded_rectangle(
        [(0, 0), (PLATE_W-1, PLATE_H-1)],
        radius=CORNER_RADIUS, outline=BLACK, width=2
    )

    # Bande bleue gauche (EU)
    draw.rectangle([2, 2, BAND_W, PLATE_H-3], fill=BLUE_EU)
    # Lettre F
    try:
        font_small = ImageFont.truetype(font_path, 18)
    except:
        font_small = ImageFont.load_default()
    draw.text((BAND_W//2, PLATE_H - 25), 'F', fill=WHITE, font=font_small, anchor='mm')
    # Etoiles EU (simplifie : un cercle de points)
    import math
    cx, cy = BAND_W // 2, 25
    for i in range(12):
        angle = math.radians(i * 30 - 90)
        sx = cx + int(12 * math.cos(angle))
        sy = cy + int(12 * math.sin(angle))
        draw.ellipse([sx-1, sy-1, sx+1, sy+1], fill=YELLOW_STARS)

    # Bande bleue droite (region)
    draw.rectangle([PLATE_W - BAND_W - 2, 2, PLATE_W - 3, PLATE_H - 3], fill=BLUE_REGION)
    # Numero de region (aleatoire)
    region = f"{random.randint(1, 95):02d}"
    draw.text((PLATE_W - BAND_W//2 - 2, PLATE_H//2 + 10), region, fill=WHITE, font=font_small, anchor='mm')

    # Texte principal
    try:
        font_main = ImageFont.truetype(font_path, 62)
    except:
        font_main = ImageFont.load_default()

    text_x = (BAND_W + PLATE_W - BAND_W) // 2
    text_y = PLATE_H // 2
    draw.text((text_x, text_y), texte, fill=BLACK, font=font_main, anchor='mm')

    return img, texte


# Test
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 5))
for ax in axes.flat:
    img, texte = generer_plaque()
    ax.imshow(np.array(img))
    ax.set_title(texte, fontsize=12)
    ax.axis('off')
plt.suptitle('Plaques SIV generees', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Augmentations realistes

In [ ]:
from PIL import ImageFilter, ImageEnhance

def augmenter_plaque(img):
    """Applique des augmentations realistes a une plaque nette."""
    # Rotation legere (-3 a +3 degres)
    angle = random.uniform(-3, 3)
    img = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=WHITE)

    # Perspective transform (simule la vue de cote)
    if random.random() > 0.5:
        w, h = img.size
        skew = random.uniform(-0.05, 0.05)
        coeffs = [
            1, skew, 0,
            0, 1, 0,
            skew * 0.001, 0,
        ]
        img = img.transform((w, h), Image.PERSPECTIVE, coeffs, Image.BICUBIC)

    # Luminosite variable
    brightness = random.uniform(0.7, 1.3)
    img = ImageEnhance.Brightness(img).enhance(brightness)

    # Contraste variable
    contrast = random.uniform(0.8, 1.2)
    img = ImageEnhance.Contrast(img).enhance(contrast)

    return img


# Test
fig, axes = plt.subplots(2, 4, figsize=(16, 5))
base_img, base_text = generer_plaque('AB-123-CD')
axes[0][0].imshow(np.array(base_img))
axes[0][0].set_title('Original')
axes[0][0].axis('off')
for ax in list(axes[0][1:]) + list(axes[1]):
    aug = augmenter_plaque(base_img.copy())
    ax.imshow(np.array(aug))
    ax.set_title('Augmente')
    ax.axis('off')
plt.suptitle('Augmentations realistes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Generer le dataset complet

In [ ]:
import json
from tqdm import tqdm

N_PLATES = 10_000
metadata = []

print(f"Generation de {N_PLATES} plaques...")
for i in tqdm(range(N_PLATES)):
    img, texte = generer_plaque()
    img = augmenter_plaque(img)

    filename = f"plate_{i:05d}.png"
    img.save(f"dataset/clean/{filename}")
    metadata.append({'id': i, 'filename': filename, 'text': texte})

# Sauvegarder les metadonnees
with open('dataset/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n{N_PLATES} plaques generees dans dataset/clean/")
print(f"Metadonnees dans dataset/metadata.json")
print(f"Exemple : {metadata[0]}")

In [ ]:
# Verification visuelle
fig, axes = plt.subplots(2, 5, figsize=(18, 5))
samples = random.sample(metadata, 10)
for ax, s in zip(axes.flat, samples):
    img = Image.open(f"dataset/clean/{s['filename']}")
    ax.imshow(np.array(img))
    ax.set_title(s['text'], fontsize=10)
    ax.axis('off')
plt.suptitle(f'{N_PLATES} plaques generees — echantillon', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Sauvegarder sur Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copier le dataset sur Drive
!mkdir -p /content/drive/MyDrive/deblurring_dataset
!cp -r dataset/* /content/drive/MyDrive/deblurring_dataset/
print("Dataset sauvegarde sur Google Drive dans /deblurring_dataset/")